In [ ]:
%load_ext autoreload
%autoreload 2


import time
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import odeint
from tqdm import tqdm
import plotly.graph_objects as go


import sys
sys.path.append("..")

from lunanav.sim.dynamics import lander_motion, linearize, lander_motion_inertial
from lunanav.sim.quaternion import angle_axis_to_q, quat_apply, hamilton_product, mul
from lunanav.plotting import plot_state_vector, debug_3d, plot_control_effort, plot_state_vector_combined
from lunanav.constants import R_MOON, GM_MOON, RAD_TO_DEG, DEG_TO_RAD
from lunanav.visualization import visualize_trajectory
from lunanav.control.lqr import ilqr_lander
from lunanav.sim.simulator import SimResults
from lunanav.loaders import Trajectory, save_trajectory
from lunanav.sim.generate import get_initial_rv_state, aggresive_smoothing, remove_outliers

def get_rv_costs():
    Q = np.diag([
        1e-3, 1e-3, 1e-3,    # position (m) — moderate
        1e-0, 1e-0, 1e-0,    # velocity (m/s) — higher (you want soft touchdown)

        1e-1, 1e-1, 1e-1, 1e-1,  # quaternion — penalize attitude error
        1e-2, 1e-2, 1e-2,    # angular velocity (rad/s)
    ])

    # Input weights — penalize fuel use / aggressive control
    R = np.diag([
        5e-1, 5e-1, 5e-1,    # force (N) — low to allow control authority
        1e-6, 1e-6, 1e-6,    # torque (N·m) — moderate
    ])

    # Terminal cost — heavy on final state
    QN = np.diag([
        1e2, 1e2, 1e2,       # position (z especially — want to land at altitude 0)
        1e4, 1e4, 1e4,       # velocity (want zero at touchdown)
        1e1, 1e1, 1e1, 1e1,  # quaternion (level attitude)
        1e1, 1e1, 1e1,       # angular velocity (no spinning)
    ])

    return Q,R,QN

def get_full_state_costs():
    Q = np.diag([
        1e-3, 1e-3, 1e-3,    # position (m) — moderate
        1e-0, 1e-0, 1e-0,    # velocity (m/s) — higher (you want soft touchdown)

        1e-1, 1e-1, 1e-1, 1e-1,  # quaternion — penalize attitude error
        1e-2, 1e-2, 1e-2,    # angular velocity (rad/s)
    ])

    # Input weights — penalize fuel use / aggressive control
    R = np.diag([
        5e-1,    # force (N) — low to allow control authority
        1e-6, 1e-6, 1e-6,    # torque (N·m) — moderate
    ])

    # Terminal cost — heavy on final state
    QN = np.diag([
        1e2, 1e2, 1e2,       # position (z especially — want to land at altitude 0)
        1e4, 1e4, 1e4,       # velocity (want zero at touchdown)
        1e1, 1e1, 1e1, 1e1,  # quaternion (level attitude)
        1e1, 1e1, 1e1,       # angular velocity (no spinning)
    ])

    return Q,R,QN


In [ ]:

### Define constants
n = 13  # state dimension
m = 6  # control dimension
Q,R,QN = get_rv_costs()


T = 200.0  # simulation time
dt = 0.1  # sampling time

mass_kg=50
I = jnp.diag(jnp.array([8.0, 8.0, 5.0]))

s0, r0, v0 = get_initial_rv_state(altitiude_m=20e3, downrange_angle_deg=3)
s0[0] += 50e3 # 15km off nominal X-pos
s0[3] += 5e3 # 15km off nominal X-vel
print(f"Initial state: {s0}")


s_goal = np.array([
    0,0,R_MOON, 
    0,0,0,
    1,0,0,0,0,0,0])


####################################################################################################
#                                       For initial position iLQR
####################################################################################################

# Initialize continuous-time and discretized dynamics
# t: float, state: jnp.ndarray, disturbances: jnp.ndarray, mass_kg: float, I: np.ndarray):
def next_state_wrapper(s, u):
    force_I = u[0:3]
    torque_B = u[3:6]
    return lander_motion_inertial(s, force_I, torque_B, dt, mass_kg, I) # mass and I from earlier


print("Computing iLQR solution ... ", end="", flush=True)
start = time.time()
t = np.arange(0.0, T+dt, dt)
N = t.size - 1
s_bar, u_bar, Y, y = ilqr_lander(next_state_wrapper, s0, s_goal, N, Q, R, QN, max_iters=20)
print("done! ({:.2f} s)".format(time.time() - start), flush=True)

####################################################################################################
#                                       Plotting results
####################################################################################################

moon_offset = np.array([0, 0, R_MOON])


print("Final position: ", s_bar[-1,0:3] - moon_offset)
print("Final velocity: ", s_bar[-1,3:6])
# u_bar =  np.concatenate([u_bar, [u_bar[-1]]]) # Pad at end to match state trajectory length
force_bar = u_bar[:,0:3]
torque = u_bar[:,3:6]

F_norms: np.ndarray = np.linalg.norm(force_bar, axis=1)
Tau_norms = np.linalg.norm(torque, axis=1)
print("Max control force: ", np.max(F_norms))
print("Max control torque: ", np.max(Tau_norms))


# visualize_trajectory(s_bar, t, dt, offset = moon_offset, downsample_rate=5, moon_resolution = 35).show()

# breakpoint()

# plot_state_vector(t, s_bar[:,0:3] - np.tile(moon_offset, (t.size, 1)), s_bar[:,3:6], s_bar[:,10:13])
# plot_control_effort(t[1:], u_bar[:,0:3], u_bar[:,3:6])

# breakpoint()


In [ ]:
# plot_state_vector(t, s_bar[:,0:3] - np.tile(moon_offset, (t.size, 1)), s_bar[:,3:6], s_bar[:,10:13])
plot_state_vector_combined(t, s_bar[:,0:3] - np.tile([0,0,R_MOON],(N+1,1)), s_bar[:,3:6], s_bar[:,10:13], title="State Vector")

In [ ]:
visualize_trajectory(s_bar, t, dt, offset = moon_offset, downsample_rate=5, moon_resolution = 35, show_lander=False).show()


In [ ]:
####################################################################################################
#                Turning position iLQR s_bar into a trajectory tracking for attitude iLQR
####################################################################################################

# Angle with respect to vertical
theta = [np.arctan2(np.sqrt(F[0]**2 + F[1]**2), F[2]) for F in force_bar] # angle between body z-axis and inertial z-axis
q_rots = [angle_axis_to_q(theta_k, [1,0,0]) for theta_k in theta] # desired quaternion from angle

# ---------------------------------------- Force ----------------------------------------
u_bar_tracking = np.zeros_like(u_bar[:,:4])
u_bar_tracking[:,0] = F_norms # feedforward force from position iLQR, but no torque feedforward


omega = np.zeros_like(theta)
for i in range(N-1):
    omega[i] = -(theta[i+1] - theta[i])/dt # in -x direction (just for this trajectory) TODO change?
alpha = np.diff(omega, axis=0) / dt

In [ ]:

# # ---------------------------------------- w ----------------------------------------
# omega_shaved = aggresive_smoothing(omega, 
#                             [
#                              (1824, 1830),
#                              (1943, 1986),
#                              ])
# omega_shaved[-1] = omega_shaved[-2]

# fig = go.Figure()
# fig.add_trace(go.Scatter(y=omega_shaved, mode='lines', name='omega'))
# fig.show()

In [ ]:

# # ---------------------------------------- alpha ----------------------------------------
# alpha_shaved = aggresive_smoothing(alpha, 
#                             [
#                              (1824, 1827),
#                              (1942, 1980),
#                              (1025, 1040),
#                              (1121, 1133),
#                              (1220, 1223),
#                              (679, 683),
#                              (122, 125),
#                              ])
# alpha_shaved[-1] = alpha_shaved[-2]

# fig = go.Figure()
# fig.add_trace(go.Scatter(y=alpha_shaved, mode='lines', name='omega'))
# fig.show()


In [ ]:
def quat_between(a, b):
    """Minimal rotation quaternion [w,x,y,z] that takes unit vec a to unit vec b."""
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    dot = float(np.clip(np.dot(a, b), -1, 1))

    if dot > 1 - 1e-9:   # already aligned
        return np.array([1., 0., 0., 0.])
    if dot < -1 + 1e-9:  # anti-parallel: 180° around any perp axis
        perp = np.array([1., 0., 0.]) if abs(a[0]) < 0.9 else np.array([0., 1., 0.])
        axis = np.cross(a, perp); axis /= np.linalg.norm(axis)
        return np.array([0., *axis])

    # half-angle trick: q = [1+cos, cross] normalised
    xyz = np.cross(a, b)
    q = np.array([1 + dot, *xyz])
    return q / np.linalg.norm(q)


def orientation_from_thrust(F_inertial, torque_inertial):
    """
    Compute body quaternions so that body +z tracks the inertial thrust direction,
    then return forces and torques expressed in that body frame.

    Args:
        F_inertial      [N, 3]  thrust vectors in inertial frame
        torque_inertial [N, 3]  torques in inertial frame

    Returns:
        q_arr       [N, 4]  desired quaternions [w,x,y,z]
        F_body      [N, 3]  force in body frame  (should be ~[0,0,|F|])
        tau_body    [N, 3]  torque in body frame
    """
    n = len(F_inertial)
    q_arr   = np.zeros((n, 4))
    F_body  = np.zeros((n, 3))
    tau_body = np.zeros((n, 3))

    thrust_axis = np.array([0., 0., 1.])

    for i in range(n):
        F = F_inertial[i]
        F_norm = np.linalg.norm(F)

        if F_norm < 1e-10:
            q = q_arr[i - 1] if i > 0 else np.array([1., 0., 0., 0.])
        else:
            q = quat_between(thrust_axis, F / F_norm)

        # enforce continuity — quaternion double-cover means q and -q are the
        # same rotation, but -q causes a flip in the plotted trajectory
        if i > 0 and np.dot(q, q_arr[i - 1]) < 0:
            q = -q

        q_arr[i] = q

        # rotate inertial → body:  v_body = conj(q) ⊗ v ⊗ q
        q_conj = np.array([q[0], -q[1], -q[2], -q[3]])
        F_body[i]   = np.array(quat_apply(q_conj, F_inertial[i]))
        tau_body[i] = np.array(quat_apply(q_conj, torque_inertial[i]))

    return q_arr, F_body, tau_body

In [ ]:
def omega_from_q(q_arr, dt):
    """Body-frame angular velocity from quaternion array via central differences."""
    n = len(q_arr)
    omega = np.zeros((n, 3))

    for i in range(n):
        if i == 0:
            q_dot = (q_arr[1] - q_arr[0]) / dt
        elif i == n - 1:
            q_dot = (q_arr[-1] - q_arr[-2]) / dt
        else:
            q_dot = (q_arr[i + 1] - q_arr[i - 1]) / (2 * dt)

        # omega_body = 2 * conj(q) ⊗ q_dot, take xyz part
        q = q_arr[i]
        q_conj = np.array([q[0], -q[1], -q[2], -q[3]])
        omega_quat = 2 * np.array(mul(q_conj, q_dot))
        omega[i] = omega_quat[1:4]  # drop the w component

    return omega


def alpha_from_omega(omega, dt):
    """Angular acceleration by central differences."""
    alpha = np.zeros_like(omega)
    alpha[1:-1] = (omega[2:] - omega[:-2]) / (2 * dt)
    alpha[0]    = (omega[1]  - omega[0])   / dt
    alpha[-1]   = (omega[-1] - omega[-2])  / dt
    return alpha


def torque_from_kinematics(omega, alpha, I):
    """Euler's equations: τ = I·α + ω × (I·ω)"""
    tau = np.zeros_like(omega)
    for i in range(len(omega)):
        tau[i] = I @ alpha[i] + np.cross(omega[i], I @ omega[i])
    return tau

In [ ]:
q_arr, F_body, _ = orientation_from_thrust(force_bar, np.zeros((N, 3)))  # torque placeholder

omega  = omega_from_q(q_arr, dt)
alpha  = alpha_from_omega(omega, dt)  # or use your existing alpha
tau_body = torque_from_kinematics(omega, alpha, I)

In [ ]:
plot_control_effort(t[:-1], F_body, tau_body)

In [ ]:

# ---------------------------------------- w ----------------------------------------
tau_shaved_x = aggresive_smoothing(tau_body[:,0],
                            [
                             (679, 684),
                             (1025, 1044),
                             (1121, 1134),
                             (1219, 1226),
                             (1818, 1830),
                             (1938, 1993),
                             ])
tau_shaved_x[-1] = tau_shaved_x[-2]

fig = go.Figure()
fig.add_trace(go.Scatter(y=tau_shaved_x, mode='lines', name='omega'))
fig.show()

In [ ]:
# ---------------------------------------- w ----------------------------------------
tau_shaved_y = aggresive_smoothing(tau_body[:,1],
                            [
                             (679, 684),
                             (1025, 1044),
                             (1121, 1134),
                             (1219, 1226),
                             (1818, 1830),
                             (1938, 1983),
                             ])
tau_shaved_y[-1] = tau_shaved_y[-2]

fig = go.Figure()
fig.add_trace(go.Scatter(y=tau_shaved_y, mode='lines', name='omega'))
fig.show()

In [ ]:
# ---------------------------------------- w ----------------------------------------
tau_shaved_z = aggresive_smoothing(tau_body[:,2],
                            [
                            #  (679, 684),
                            #  (1025, 1044),
                            #  (1121, 1134),
                            #  (1219, 1226),
                            #  (1818, 1830),
                             (1938, 1983),
                             ])
tau_shaved_z[-1] = tau_shaved_z[-2]

fig = go.Figure()
fig.add_trace(go.Scatter(y=tau_shaved_z, mode='lines', name='omega'))
fig.show()

In [ ]:
tau_body = np.vstack((tau_shaved_x, tau_shaved_y, tau_shaved_z)).T

In [ ]:
# force = F_norms.reshape(-1, 1) * np.array([0, 0, 1])
# torque_norm = alpha_shaved * I[0,0] # X torque
# torque = torque_norm.reshape(-1, 1) * np.array([[1,0,0]]) # only X torque for this trajectory

In [ ]:
plot_control_effort(t[:-1], F_body, tau_body)

In [ ]:
TRAJ_FILE = "data/trajectories/ilqr_skew.json"

save_trajectory(Trajectory(
    s_bar=s_bar, u_bar=u_bar, dt=dt, T=T, nsteps=N,
    state0=s0, mass_kg=mass_kg, I=np.array(I),
    t=t, force=F_body, torque=tau_body,
), TRAJ_FILE)